<a href="https://colab.research.google.com/github/Aggarwalmansi/GENAI/blob/main/AgentLangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain langchain-community langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.8
    Uninstalling langgraph-prebuilt-1.0.8:
      Successfully uninstalled langgraph-prebuilt-1.0.8
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.4
  

In [2]:
!pip install -q langchain-classic

In [3]:
import os
from getpass import getpass

# Set up the Groq API Key
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

from langchain_groq import ChatGroq

# Initialize the Groq LLM
# llama-3.3-70b-versatile is a highly capable and fast free model on Groq
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)
print(":white_check_mark: Groq LLM ready!")

Enter your Groq API Key: ··········
:white_check_mark: Groq LLM ready!


### lcel - chain  , we are using pipe operator | to define the chain - this is lcel .

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Simple chain for itinerary generation
prompt = ChatPromptTemplate.from_template(
    "You are a world-class travel planner. Create a 3-day itinerary for {destination} for a {traveler_type} traveler."
)

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"destination": "Lucknow", "traveler_type": "veg foodie"})
print(result)

Lucknow, the City of Nawabs, is a treasure trove of rich history, vibrant culture, and delectable cuisine. As a world-class travel planner, I've crafted a 3-day itinerary for a veg foodie traveler to explore the best of Lucknow. Here's your culinary journey:

**Day 1:**

* Morning: Start with a visit to the iconic **Chhota Imambara**, a beautiful mosque with stunning architecture. Take a stroll around the nearby **Bhool Bhulaiya**, a labyrinthine maze of narrow alleys and ornate buildings.
* Breakfast: Head to **Idris Biryani** (they have a veg biryani option) or **Tunday Kebabi** (try their veg kebabs) for a flavorful start to the day. Don't forget to try some **Lucknowi chaat** at a local street food stall.
* Lunch: Visit **Dastarkhwan**, a popular restaurant serving authentic Lucknowi cuisine. Try their veg dishes like **Dal Makhani**, **Saag Paneer**, or **Vegetable Biryani**.
* Evening: Explore the **Hazratganj Market**, a bustling shopping hub with a variety of street food option

In [21]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Prompt with history
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are SmartVoyage, a helpful AI travel planner."),
    ("placeholder", "{history}"),
    ("human", "{input}")
])

chain = prompt | llm | StrOutputParser()

# Memory store
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Wrap chain with memory
conversational_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# Test memory
print("=== First message ===")
print(conversational_chain.invoke(
    {"input": "Hi, I want to plan a trip to germany next month."},
    config={"configurable": {"session_id": "lab_student3"}}
))

print("\n=== Second message (memory works!) ===")
print(conversational_chain.invoke(
    {"input": "Make it budget-friendly and foodie."},
    config={"configurable": {"session_id": "lab_student3"}}
))

=== First message ===
Germany is a wonderful destination. There's so much to see and experience, from vibrant cities like Berlin and Munich to picturesque countryside and rich history.

To help me plan your trip, could you please provide some more details? 

1. How long do you plan to stay in Germany?
2. What are your interests? (e.g. history, culture, food, outdoor activities, etc.)
3. Are there any specific cities or regions you'd like to visit?
4. What's your approximate budget for the trip?
5. Are you traveling solo, with friends, or with family?
6. Do you have any preferred accommodation types (e.g. hotel, hostel, Airbnb)?

Let's start planning your German adventure!

=== Second message (memory works!) ===
A budget-friendly and foodie trip to Germany sounds like a great idea. Here's a rough outline to get you started:

**When to Go:**
Since you're planning to visit next month, I assume you're looking at April. The weather can be quite pleasant, with mild temperatures and fewer tou

In [26]:
pip install requests

In [33]:
from langchain_core.tools import tool
import requests
@tool
def get_destination_facts(destination: str) -> str:
    """Return key facts, best time to visit, and top attractions for a destination."""
    facts = {
        "paris": "Eiffel Tower, Louvre, best in spring. Romantic atmosphere.",
        "tokyo": "Shibuya, Mount Fuji, anime culture. Best in cherry blossom season.",
        "new york": "Statue of Liberty, Times Square, Central Park. Iconic city that never sleeps."
    }
    return facts.get(destination.lower(), "Sorry, I don't have info on that destination yet.")

@tool
def calculate_travel_budget(days: int, people: int, budget_level: str) -> str:
    """Simple budget calculator. budget_level = low/medium/luxury"""
    multipliers = {"low": 100, "medium": 200, "luxury": 400}
    total = days * people * multipliers.get(budget_level.lower(), 200)
    return f"Estimated total budget: ${total} (for {days} days, {people} people, {budget_level} level)"
@tool
def get_destination_weather(destination: str) -> str:
    """Return the current weather for a destination."""
    api_key = "71f04e73ee62c6c521ee6e01a53fff64"
    url = f"http://api.openweathermap.org/data/2.5/weather?q={destination}&appid={api_key}&units=metric"
    response = requests.get(url)

    if response.status_code != 200:
        return "Could not fetch weather data."

    data = response.json()
    weather = data["weather"][0]["description"]
    temp = data["main"]["temp"]

    return f"The current weather in {destination} is {weather} with temperature {temp}°C."

tools = [get_destination_facts, calculate_travel_budget,get_destination_weather]
print(":white_check_mark: Tools created!")

:white_check_mark: Tools created!


In [34]:
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# Agent prompt (same as before)
system_prompt = """You are SmartVoyage, a helpful AI travel planner.
Use the provided tools when needed. Be friendly, concise, and creative.
Always ask clarifying questions if information is missing."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# Create the agent
agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)

# Create the executor (the "engine" that runs the agent)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=False,           # shows thinking steps
    handle_parsing_errors=True
)

# Test it
response = agent_executor.invoke({"input": "Plan a 5-day budget trip to Paris for 2 people also tell its weather"})
print(response["output"])

Based on the budget calculation, a 5-day trip to Paris for 2 people on a low budget would cost approximately $1000. 

As for the weather, I was unable to fetch the current weather data for Paris. However, I can suggest checking a weather forecast website or app for the most up-to-date information.

To plan your trip, I can suggest some free or low-cost activities to do in Paris, such as visiting the Louvre Museum, taking a stroll along the Seine River, or exploring the Montmartre neighborhood. You can also consider purchasing a Paris Museum Pass, which grants you entry to many of the city's top attractions.

Let me know if you have any other questions or if there's anything else I can help you with!
